# C3-ft epilogue diagnostics — cache + domain probe + NCM/kNN + t-SNE

**Declared epilogue (owner call, 2026-07-19): no decision feeds on
these numbers.** The SupCon chapter is closed by six instruments; this
session runs the same instruments on the C3-ft encoder (`best.ckpt`,
epoch 4) for completeness of the record. One session does everything:
cache -> domain probe -> NCM/kNN -> t-SNE panel.

**Expected outcomes, written BEFORE running** (§0 discipline — if any
row surprises, stop and bring it to the team):
- domain probe: every domain target at its majority baseline (5th
  replication — both parents of this encoder, the C3 init and the CE
  objective, have unreadable domain); `y` control high (seen traces);
- NCM/kNN: at or below the ~0.82 SupCon representation ceiling;
- t-SNE: the L/S/E chain of C3 likely still visible — the best sits at
  epoch 4 of warmup (LR 8e-5), the encoder barely moved from init.

**Scope note:** the t-SNE here is a DIAGNOSTIC panel. The §9 report
figure stays C1 vs C3 as pinned — promoting a C3-ft panel into the
report figure would need a team call, not this notebook.

Train/val features only, no test contact (§0.7).

## Environment setup

Full preamble incl. data staging (the cache step runs the encoder over
the real staged data). GPU recommended for the cache; the diagnostics
after it are CPU-fine.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import torch
import zipfile

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"

cwd = Path.cwd().resolve()
if (cwd / "sharp_har").exists():
    REPO_DIR = cwd
elif (cwd.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent
elif (cwd.parent.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent.parent
else:
    REPO_DIR = Path("/content/sharp-har")
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

# After the clone, so the file actually exists on a fresh runtime.
!pip install -q -r {REPO_DIR}/requirements.txt

sys.path.insert(0, str(REPO_DIR))

from google.colab import drive
from sharp_har.utils import read_yaml

paths_cfg = read_yaml(REPO_DIR / "configs" / "paths.yaml")
drive_root = Path(paths_cfg["drive_root"])
stage_dir = Path(paths_cfg["stage_dir"])
CKPT_ROOT = Path(paths_cfg["ckpt_root"])

drive.mount("/content/drive")  # idempotent

if not (stage_dir.exists() and any(stage_dir.rglob("*.txt"))):
    stage_dir.mkdir(parents=True, exist_ok=True)
    for zip_name in paths_cfg["zips"]:
        src = drive_root / zip_name
        dst = Path("/content") / zip_name
        print(f"copying {src} -> {dst}")
        subprocess.run(["cp", str(src), str(dst)], check=True)
        with zipfile.ZipFile(dst) as zf:
            zf.extractall(stage_dir)

print("Repo dir:", REPO_DIR)
print("GPU available:", torch.cuda.is_available())
print("Checkpoint root:", CKPT_ROOT)

## 1. Cache C3_ft train+val features

Same path as every probe session (`harness.cache_features`, frozen
encoder, no augmentation). Safe to re-run (overwrites with an
identical extraction).

In [ ]:
import numpy as np
from sharp_har.harness import cache_features

CKPT = CKPT_ROOT / "C3_ft" / "best.ckpt"
P2_SPLIT = REPO_DIR / "splits" / "p2_lab.json"
assert CKPT.exists(), f"{CKPT} not found - C3_ft run folder missing?"

EXPECTED = {"train": 53400, "val": 1396}
cache = {}
for set_name in ("train", "val"):
    cache[set_name] = cache_features(
        CKPT, P2_SPLIT, set_name, stage_dir=stage_dir, repo_dir=REPO_DIR,
    )
    d = np.load(cache[set_name], allow_pickle=False)
    n, dim = d["features"].shape
    assert n == EXPECTED[set_name] and dim == 256, f"{set_name}: {n}x{dim}, expected {EXPECTED[set_name]}x256"
    print(f"{set_name}: {n} samples, d={dim} -> {cache[set_name]}")

## 2. Domain probe (5th replication, expected: all deltas ~ 0)

The promoted instrument (`diagnostics.domain_probe`) — identical
recipe and inner split as the archived C1/C2/C3/C1_s6out sessions.

In [ ]:
from sharp_har.diagnostics import domain_probe

rows = domain_probe(cache["train"], "C3_ft")

print()
print(f"{'target':12s} {'acc':>7s} {'baseline':>9s} {'delta':>8s} {'macroF1':>8s}")
for r in rows:
    print(f"{r['target']:12s} {r['eval_accuracy']:7.3f} {r['majority_baseline']:9.3f} "
          f"{r['delta']:+8.3f} {r['macro_f1']:8.3f}")
print("\nreference (archived sessions), y-control then ar_set delta:")
print("C1 1.000 / -0.090 | C2 0.893 / -0.030 | C3@40 0.995 / (grid, ~0) | C1_s6out 0.870 / +0.011")

## 3. NCM / kNN (expected: at or below the ~0.82 ceiling)

Promoted scorers (`diagnostics.ncm_scores`/`knn_scores`), fused with
the same harness path as every stream.

In [ ]:
from sharp_har.diagnostics import l2norm, ncm_scores, knn_scores
from sharp_har.harness import fuse_windows, macro_f1
from sharp_har.probe import majority_baseline

train = np.load(cache["train"], allow_pickle=False)
val = np.load(cache["val"], allow_pickle=False)
train_x, train_y = l2norm(train["features"].astype(np.float64)), train["y"]
val_x, val_y = l2norm(val["features"].astype(np.float64)), val["y"]
n_classes = len(train["labels"])
tid, ws = val["trace_id"].astype(object), val["window_start"]

for clf_name, scores in (
    ("NCM", ncm_scores(train_x, train_y, n_classes, val_x)),
    ("kNN", knn_scores(train_x, train_y, n_classes, val_x)),
):
    fused = fuse_windows(scores, val_y, tid, ws)
    acc = float((fused["y_pred"] == fused["y_true"]).mean())
    f1 = macro_f1(fused["y_true"], fused["y_pred"])
    print(f"C3_ft [{clf_name}]: acc {acc:.4f}, macro-F1 {f1:.4f} "
          f"(majority baseline {majority_baseline(fused['y_true']):.4f})")

print("\nreference: C3_ft end-to-end best 0.8183 | C3-lin 0.8190 | "
      "C3 NCM 0.7178 / kNN 0.8047 | C1 linear 0.8835")

## 4. t-SNE — C1 vs C3 vs C3-ft (diagnostic panel)

Same declared recipe (`viz.plot_embeddings`). Reads: did 4 warmup
epochs of CE un-chain C3's L/S/E structure? Expected: mostly not.
**Not the §9 report figure** (scope pinned C1 vs C3 — a promotion
needs a team call).

In [ ]:
from sharp_har.viz import plot_embeddings

c3_sel = json.loads((CKPT_ROOT / "C3" / "phase_b_selection.json").read_text())
c3_stem = Path(c3_sel["selected_checkpoint"]).stem

FEATURE_CACHES = {
    "C1": CKPT_ROOT / "C1" / "features_best_train.npz",
    "C3": CKPT_ROOT / "C3" / f"features_{c3_stem}_train.npz",
    "C3-ft": cache["train"],
}
for label, p in FEATURE_CACHES.items():
    assert Path(p).exists(), f"{label}: missing {p}"

plot_embeddings(FEATURE_CACHES, save_path=REPO_DIR / "reports" / "embeddings_c1_c3_c3ft_diagnostic.png");

## Archiving

Download the executed copy (with outputs and the figure) and commit it
verbatim over this file, then fill the Outcome cell in this folder's
README + a STATUS line, same commit. The PNG under `reports/` is a
diagnostic asset, not the §9 figure.